In [1]:
import numpy as np

In [2]:
def rastrigin(x):
    A = 10
    n = len(x)
    
    return A * n + sum([x[i] ** 2 - A * np.cos(2 * np.pi * x[i]) for i in range(n)])

In [3]:
def rosenbrock(x):
    a = 1
    b = 100
    
    return (a - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2

In [4]:
n_dims = 3

bounds = np.array([(-5.12, 5.12)] * n_dims)
bounds

array([[-5.12,  5.12],
       [-5.12,  5.12],
       [-5.12,  5.12]])

In [5]:
class Particle:
    global_best_pos = None
    global_best_val = float('inf')
    
    def __init__(self, bounds, objective, w, c1, c2):
        # Pošto su granice iste za sve koordinate, početnu poziciju možemo naći na sledeći način
        self.pos = np.random.uniform(low = bounds[0][0], high = bounds[0][1], size = len(bounds))
        
        # Da granice nisu bile iste za sve koordinate, poziciju bismo izračunali kao:
        # self.pos = np.random.uniform(low = bounds[:, 0], high = bounds[:, 1], size = len(bounds))
        
        self.curr_val = objective(self.pos)
        self.objective = objective
        
        self.best_val = self.curr_val
        self.best_pos = self.pos.copy()
        
        random_point = np.random.uniform(low = bounds[0][0], high = bounds[0][1], size = len(bounds))
        # Analogno poziciji, da granice nisu bile iste za sve koordinate, poziciju nasumične tačke izračunali bismo
        # random_point = np.random.uniform(low = bounds[:, 0], high = bounds[:, 1], size = len(bounds))
        self.velocity = random_point - self.pos
        
        if self.best_val < Particle.global_best_val:
            Particle.global_best_val = self.best_val
            Particle.global_best_pos = self.best_pos.copy()
            
        self.w = w
        self.c1 = c1
        self.c2 = c2
        
        self.bounds = bounds
        
    def move(self):
        # Najpre ažuriramo poziciju čestice koristeći brzinu iz prethodne iteracije.
        self.pos += self.velocity
        
        # Pomeranje čestice moglo je dovesti do toga da ona "iskoči" iz dozvoljenog opsega, pa se moramo osigurati
        # da će se i njena nova pozicija takođe nalaziti unutar dozvoljenih vrednosti.
        self.pos = np.array([np.clip(self.pos[i], self.bounds[i][0], self.bounds[i][1]) for i in range(len(self.bounds))])
        
        self.curr_val = self.objective(self.pos)
        if self.curr_val < self.best_val:
            self.best_pos = self.pos.copy()
            self.best_val = self.curr_val
            
            if Particle.global_best_val > self.best_val:
                Particle.global_best_val = self.best_val
                Particle.global_best_pos = self.best_pos.copy()
                
        # inicijalizujemo novu brzinu čestice za sledeću iteraciju
        inertia = self.velocity * self.w
        cognitive_inteligence = self.c1 * (self.best_pos - self.pos) * np.random.random(size = len(self.pos))
        social_inteligence = self.c2 * (Particle.global_best_pos - self.pos) * np.random.random(size = len(self.pos))
        
        self.velocity = inertia + cognitive_inteligence + social_inteligence

In [6]:
def pso(num_iters, swarm_size, bounds, objective, w, c1, c2):
    # NAPOMENA: promenljive global_best_pos i global_best_val su zajedničke za sve instance klase Particle i kao 
    # takve, one se inicijalizuju samo jednom. To znači da bi, u slučaju pokretanja ove funkcije za optimizaciju 
    # različitih problema, pomenute promenljive imale vrednost koju su imale na kraju prvog rešenog problema, što
    # može dovesti do grešaka kasnije. Stoga, možemo pre svakog pokretanja funkcije postaviti podrazumevane vrednosti
    # ovih promenljivih.
    Particle.global_best_pos = None
    Particle.global_best_val = float('inf')
    
    swarm = [Particle(bounds, objective, w, c1, c2) for i in range(swarm_size)]
    
    for _ in range(num_iters):
        for particle in swarm:
            particle.move()
    
    return Particle.global_best_pos, Particle.global_best_val

In [7]:
print(pso(1000, 30, bounds, rastrigin, 0.5, 1, 2))

(array([ 1.06846422e-09, -1.34498734e-09,  1.14498176e-09]), np.float64(0.0))


In [8]:
n_dims = 2

bounds = np.array([(-5.12, 5.12)] * n_dims)
bounds

array([[-5.12,  5.12],
       [-5.12,  5.12]])

In [9]:
print(pso(1000, 30, bounds, rosenbrock, 0.5, 1, 2))

(array([1., 1.]), np.float64(0.0))
